In [1]:
!nvidia-smi
!nvcc --version
!command -v compute-sanitizer || true
!python --version

Fri Aug 21 14:31:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
import subprocess
import json
from pathlib import Path
from datetime import datetime, timezone

os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ.get("PATH", "")

def run(command):
    return subprocess.run(
        command,
        shell=True,
        text=True,
        capture_output=True,
        check=False,
    )

checks = {
    "gpu": run("nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader").stdout.strip(),
    "nvcc": run("nvcc --version").stdout.strip(),
    "compute_sanitizer": run("compute-sanitizer --version").stdout.strip(),
    "python": run("python --version").stdout.strip(),
}

print(json.dumps(checks, indent=2))

Path("env.json").write_text(json.dumps({
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    **checks
}, indent=2))

{
  "gpu": "Tesla T4, 580.82.07, 15360 MiB",
  "nvcc": "nvcc: NVIDIA (R) Cuda compiler driver\nCopyright (c) 2005-2025 NVIDIA Corporation\nBuilt on Fri_Feb_21_20:23:50_PST_2025\nCuda compilation tools, release 12.8, V12.8.93\nBuild cuda_12.8.r12.8/compiler.35583870_0",
  "compute_sanitizer": "NVIDIA (R) Compute Sanitizer\nCopyright (c) 2020-2025 NVIDIA Corporation\nVersion 2025.1.0.0 (build 35583870) (public-release)",
  "python": "Python 3.12.13"
}


510

In [6]:
%%writefile vector_add.cu
#include <cuda_runtime.h>

#include <cmath>
#include <iostream>
#include <vector>

__global__ void vector_add(const float* a, const float* b, float* c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        c[i] = a[i] + b[i];
    }
}

int main() {
    constexpr int n = 1024;
    constexpr size_t bytes = n * sizeof(float);

    std::vector<float> a(n, 2.0f);
    std::vector<float> b(n, 3.0f);
    std::vector<float> c(n, 0.0f);

    float* da = nullptr;
    float* db = nullptr;
    float* dc = nullptr;

    cudaMalloc(&da, bytes);
    cudaMalloc(&db, bytes);
    cudaMalloc(&dc, bytes);

    cudaMemcpy(da, a.data(), bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(db, b.data(), bytes, cudaMemcpyHostToDevice);

    vector_add<<<(n + 255) / 256, 256>>>(da, db, dc, n);

    cudaError_t error = cudaDeviceSynchronize();
    if (error != cudaSuccess) {
        std::cerr << "CUDA error: " << cudaGetErrorString(error) << '\n';
        return 1;
    }

    cudaMemcpy(c.data(), dc, bytes, cudaMemcpyDeviceToHost);

    cudaFree(da);
    cudaFree(db);
    cudaFree(dc);

    for (float value : c) {
        if (value != 5.0f) {
            std::cerr << "Incorrect vector-add result\n";
            return 1;
        }
    }

    std::cout << "vector-add: PASS\n";
    return 0;
}
!nvcc -std=c++20 -O0 -lineinfo -arch=sm_75 vector_add.cu -o vector_add
!./vector_add

Overwriting vector_add.cu


In [7]:
%%writefile reduction_test.cu
#include <cuda_runtime.h>

#include <algorithm>
#include <cmath>
#include <cstdlib>
#include <fstream>
#include <iomanip>
#include <iostream>
#include <string>
#include <vector>

__global__ void reduce_sum_kernel(
    const float* input,
    float* output,
    int n,
    bool faulty
) {
    extern __shared__ float shared[];

    int tid = threadIdx.x;
    float local_sum = 0.0f;

    for (int i = tid; i < n; i += blockDim.x) {
        if (!faulty || i < n - 1) {
            local_sum += input[i];
        }
    }

    shared[tid] = local_sum;
    __syncthreads();

    for (int stride = blockDim.x / 2; stride > 0; stride /= 2) {
        if (tid < stride) {
            shared[tid] += shared[tid + stride];
        }
        __syncthreads();
    }

    if (tid == 0) {
        output[0] = shared[0];
    }
}

std::vector<float> make_input(int n, int pattern) {
    std::vector<float> values(n);

    for (int i = 0; i < n; ++i) {
        if (pattern == 0) {
            values[i] = 1.0f;
        } else if (pattern == 1) {
            values[i] = (i % 2 == 0) ? 1.0f : -1.0f;
        } else {
            values[i] = static_cast<float>((i % 7) - 3);
        }
    }

    return values;
}

double reference_sum(const std::vector<float>& values) {
    double result = 0.0;

    for (float value : values) {
        result += static_cast<double>(value);
    }

    return result;
}

void write_failure(
    const std::vector<float>& input,
    int pattern,
    int block_size,
    double expected,
    float actual,
    const cudaDeviceProp& device
) {
    std::ofstream file("failure.json");

    file << std::setprecision(9);
    file << "{\n";
    file << "  \"schema\": \"vanta.failure.v0\",\n";
    file << "  \"kernel\": \"reduce_sum\",\n";
    file << "  \"implementation\": \"faulty_tail\",\n";
    file << "  \"input\": {\n";
    file << "    \"dtype\": \"float32\",\n";
    file << "    \"shape\": [" << input.size() << "],\n";
    file << "    \"values\": [";

    for (size_t i = 0; i < input.size(); ++i) {
        if (i > 0) {
            file << ", ";
        }
        file << input[i];
    }

    file << "]\n";
    file << "  },\n";
    file << "  \"launch\": {\n";
    file << "    \"block_threads\": " << block_size << ",\n";
    file << "    \"grid_blocks\": 1\n";
    file << "  },\n";
    file << "  \"environment\": {\n";
    file << "    \"gpu\": \"" << device.name << "\",\n";
    file << "    \"compute_capability\": \""
         << device.major << "." << device.minor << "\"\n";
    file << "  },\n";
    file << "  \"observed\": {\n";
    file << "    \"kind\": \"correctness_mismatch\",\n";
    file << "    \"pattern\": " << pattern << ",\n";
    file << "    \"expected\": " << expected << ",\n";
    file << "    \"actual\": " << actual << "\n";
    file << "  }\n";
    file << "}\n";
}

int main(int argc, char** argv) {
    if (argc != 2 || (std::string(argv[1]) != "correct" &&
                      std::string(argv[1]) != "faulty")) {
        std::cerr << "Usage: ./reduction_test correct|faulty\n";
        return 2;
    }

    const bool faulty = std::string(argv[1]) == "faulty";

    cudaDeviceProp device{};
    cudaGetDeviceProperties(&device, 0);

    std::cout << "GPU: " << device.name << '\n';
    std::cout << "Compute capability: "
              << device.major << "." << device.minor << '\n';

    const std::vector<int> sizes = {31, 32, 33, 255, 256, 257};
    const std::vector<int> block_sizes = {32, 64, 128, 256};
    const std::vector<int> patterns = {0, 1, 2};

    int mismatch_count = 0;
    int pass_count = 0;

    for (int n : sizes) {
        for (int pattern : patterns) {
            std::vector<float> input = make_input(n, pattern);
            double expected = reference_sum(input);

            for (int block_size : block_sizes) {
                float* device_input = nullptr;
                float* device_output = nullptr;

                cudaMalloc(
                    &device_input,
                    input.size() * sizeof(float)
                );
                cudaMalloc(&device_output, sizeof(float));

                cudaMemcpy(
                    device_input,
                    input.data(),
                    input.size() * sizeof(float),
                    cudaMemcpyHostToDevice
                );

                reduce_sum_kernel<<<
                    1,
                    block_size,
                    block_size * sizeof(float)
                >>>(
                    device_input,
                    device_output,
                    n,
                    faulty
                );

                cudaError_t error = cudaDeviceSynchronize();

                if (error != cudaSuccess) {
                    std::cerr << "CUDA error: "
                              << cudaGetErrorString(error) << '\n';
                    cudaFree(device_input);
                    cudaFree(device_output);
                    return 1;
                }

                float actual = 0.0f;

                cudaMemcpy(
                    &actual,
                    device_output,
                    sizeof(float),
                    cudaMemcpyDeviceToHost
                );

                cudaFree(device_input);
                cudaFree(device_output);

                double tolerance =
                    1e-5 * std::max(1.0, std::abs(expected));

                bool passed =
                    std::abs(static_cast<double>(actual) - expected)
                    <= tolerance;

                if (passed) {
                    ++pass_count;
                    std::cout << "PASS"
                              << " n=" << n
                              << " pattern=" << pattern
                              << " block=" << block_size
                              << '\n';
                } else {
                    ++mismatch_count;
                    std::cout << "MISMATCH"
                              << " n=" << n
                              << " pattern=" << pattern
                              << " block=" << block_size
                              << " expected=" << expected
                              << " actual=" << actual
                              << '\n';

                    if (faulty && mismatch_count == 1) {
                        write_failure(
                            input,
                            pattern,
                            block_size,
                            expected,
                            actual,
                            device
                        );
                    }
                }
            }
        }
    }

    std::cout << "pass_count=" << pass_count << '\n';
    std::cout << "mismatch_count=" << mismatch_count << '\n';

    if (!faulty && mismatch_count == 0) {
        std::cout << "correct kernel: PASS\n";
        return 0;
    }

    if (faulty && mismatch_count > 0) {
        std::cout << "faulty kernel correctly detected: PASS\n";
        return 0;
    }

    std::cerr << "unexpected validation result\n";
    return 1;
}

Writing reduction_test.cu


In [8]:
!nvcc -std=c++20 -O0 -lineinfo -arch=sm_75 \
    reduction_test.cu \
    -o reduction_test
!rm -f failure.json
!./reduction_test correct

GPU: Tesla T4
Compute capability: 7.5
PASS n=31 pattern=0 block=32
PASS n=31 pattern=0 block=64
PASS n=31 pattern=0 block=128
PASS n=31 pattern=0 block=256
PASS n=31 pattern=1 block=32
PASS n=31 pattern=1 block=64
PASS n=31 pattern=1 block=128
PASS n=31 pattern=1 block=256
PASS n=31 pattern=2 block=32
PASS n=31 pattern=2 block=64
PASS n=31 pattern=2 block=128
PASS n=31 pattern=2 block=256
PASS n=32 pattern=0 block=32
PASS n=32 pattern=0 block=64
PASS n=32 pattern=0 block=128
PASS n=32 pattern=0 block=256
PASS n=32 pattern=1 block=32
PASS n=32 pattern=1 block=64
PASS n=32 pattern=1 block=128
PASS n=32 pattern=1 block=256
PASS n=32 pattern=2 block=32
PASS n=32 pattern=2 block=64
PASS n=32 pattern=2 block=128
PASS n=32 pattern=2 block=256
PASS n=33 pattern=0 block=32
PASS n=33 pattern=0 block=64
PASS n=33 pattern=0 block=128
PASS n=33 pattern=0 block=256
PASS n=33 pattern=1 block=32
PASS n=33 pattern=1 block=64
PASS n=33 pattern=1 block=128
PASS n=33 pattern=1 block=256
PASS n=33 pattern=

In [10]:
!/usr/local/cuda/bin/compute-sanitizer \
    --tool memcheck \
    --error-exitcode 1 \
    ./reduction_test correct

========= COMPUTE-SANITIZER
GPU: Tesla T4
Compute capability: 7.5
PASS n=31 pattern=0 block=32
PASS n=31 pattern=0 block=64
PASS n=31 pattern=0 block=128
PASS n=31 pattern=0 block=256
PASS n=31 pattern=1 block=32
PASS n=31 pattern=1 block=64
PASS n=31 pattern=1 block=128
PASS n=31 pattern=1 block=256
PASS n=31 pattern=2 block=32
PASS n=31 pattern=2 block=64
PASS n=31 pattern=2 block=128
PASS n=31 pattern=2 block=256
PASS n=32 pattern=0 block=32
PASS n=32 pattern=0 block=64
PASS n=32 pattern=0 block=128
PASS n=32 pattern=0 block=256
PASS n=32 pattern=1 block=32
PASS n=32 pattern=1 block=64
PASS n=32 pattern=1 block=128
PASS n=32 pattern=1 block=256
PASS n=32 pattern=2 block=32
PASS n=32 pattern=2 block=64
PASS n=32 pattern=2 block=128
PASS n=32 pattern=2 block=256
PASS n=33 pattern=0 block=32
PASS n=33 pattern=0 block=64
PASS n=33 pattern=0 block=128
PASS n=33 pattern=0 block=256
PASS n=33 pattern=1 block=32
PASS n=33 pattern=1 block=64
PASS n=33 pattern=1 block=128
PASS n=33 pattern=1 

In [11]:
from google.colab import files

for filename in ["env.json", "failure.json"]:
    if Path(filename).exists():
        files.download(filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>